# Cloud-Optimized Data Preparation Workshop

## Overview
This notebook walks through converting traditional geospatial data formats into cloud-optimized alternatives that enable efficient web-based visualization via HTTP range requests.

### Data Sources Used in This Workshop

This workshop uses publicly available data:

1. **Santa Clara County Road Network (Vector)** - [RoadCenterLine Dataset](https://data.sccgov.org/Transportation/RoadCenterLine/amyq-ahrs)
   - Public transportation infrastructure data
   - Open Data portal (Socrata)
   - Available in multiple formats including shapefile

2. **Stanford Digital Repository Reference** - [World PMTiles](https://purl.stanford.edu/hf224mw4004)
   - Large-scale example of cloud-optimized data (108 GiB)
   - Demonstrates the full potential of PMTiles format
   - Live URL: `https://stacks.stanford.edu/file/druid:hf224mw4004/20231116.pmtiles`

3. **David Rumsey Historical Maps (Optional Raster)** - [Santa Clara County Maps](https://www.davidrumsey.com/luna/servlet/detail/RUMSEY~8~1~1578~170036:San-Mateo,-Santa-Cruz,-Santa-Clara)
   - Georeferenced historical maps for comparison with contemporary data
   - Example of raster data suitable for COG conversion
   - Demonstrates temporal analysis capabilities

### What We'll Do
1. **Examine source data** using GDAL command-line tools
2. **Convert vector data** (shapefiles) → FlatGeoBuf → PMTiles
3. **Convert raster data** (`santa_clara_1896.tif`) → Cloud-Optimized GeoTIFF (COG)
4. **Generate metadata** and prepare URLs for web deployment

### Key Formats
- **PMTiles**: Vector tile container with embedded index (for efficient HTTP range requests)
- **FlatGeoBuf**: Intermediate vector format supporting streaming

### Output
- `santa_clara_roads.pmtiles` - Vector tile dataset of road network
- `santa_clara_1896_cog.tif` - Cloud-optimized GeoTIFF derived from the historical map
- HTTP URLs ready for web map deployment

## Section 1: Install Dependencies and Import Libraries

In [ ]:
# Install required packages
# Run this cell first to set up your environment
# If you are running in Colab, this cell starts the single setup path
# for the notebook: it installs the command-line tools needed here.

import os
import subprocess
import sys

if 'COLAB_RELEASE_TAG' in os.environ:
    print("Detected Google Colab. Running the single setup path for this notebook...")
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'gdal-bin', 'nodejs', 'npm'], check=True)
    subprocess.run(['npm', 'install', '-g', 'tippecanoe'], check=True) # Install tippecanoe via npm

    print("✓ GDAL, NodeJS, npm, and Tippecanoe installed for Colab")
else:
    print("Running outside Colab. Install GDAL and Tippecanoe separately if needed.")

# The packages below are Python libraries, not command-line tools.
packages = [
    'geopandas',      # Vector data manipulation
    'rasterio',       # Raster data handling
    'shapely',        # Geometric operations
    'pandas',         # Data analysis
    'pyproj',         # Coordinate reference systems
]

print("Installing required Python packages...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ Python package installation complete!")
print("\nNote: on non-Colab systems, install GDAL and Tippecanoe separately if you want to run the notebook locally:")
print("  - macOS: brew install gdal")
print("  - Ubuntu: sudo apt-get install gdal-bin")
print("  - Tippecanoe: npm install -g tippecanoe")
print("  - Windows: Download from https://trac.osgeo.org/gdal/wiki/DownloadingGdalBinaries")

In [ ]:
# Colab setup: copy the workshop data into /content/data
# This cell only needs to run in Google Colab.

import os
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
GITHUB_REPO = 'https://github.com/mapninja/geo4lib_cloud_optimized.git'

# Colab sets this environment variable, so we can keep the notebook
# lightweight when it is run locally.
RUN_IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if RUN_IN_COLAB:
    # Remove any old copy so the notebook always starts from fresh data.
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)

    repo_dir = PROJECT_ROOT / '_geo4lib_cloud_optimized_repo'
    if repo_dir.exists():
        shutil.rmtree(repo_dir)

    print("Downloading the workshop repository from GitHub...")
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, str(repo_dir)], check=True)

    # Copy only the shared data folder into /content/data so the rest of
    # the notebook can use the same paths as the local repository.
    shutil.copytree(repo_dir / 'data', DATA_DIR)
    shutil.rmtree(repo_dir)

    print(f"✓ Data copied to {DATA_DIR}")
else:
    print("Running outside Colab. The notebook will use the local data/ folder in this repository.")
    print(f"Data directory: {DATA_DIR}")


In [ ]:
# Import required libraries

import os
import subprocess
import json
from pathlib import Path
from typing import Dict, Tuple

# Geospatial libraries
import geopandas as gpd
import rasterio
from rasterio.io import MemoryFile
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

# Data manipulation
import pandas as pd
from shapely.geometry import box

# System utilities
import logging
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Define project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
SCRATCH_DIR = PROJECT_ROOT / 'scratch'

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
SCRATCH_DIR.mkdir(exist_ok=True)

print("✓ All libraries imported successfully!")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## Section 2: Examine and Load Source Data

### Step 1: Inspect Shapefile with GDAL

First, we'll use GDAL command-line tools to examine the shapefile structure. This shows us:
- Layer names and feature counts
- Geometry types
- Coordinate reference system (CRS)
- Attribute fields and their data types
- Spatial extent (bounding box)

In [ ]:
# Examine the shapefile using GDAL command-line tools
# This step inspects the source data without loading it into memory

shapefile_path = DATA_DIR / 'RoadCenterLine_20260610' / 'geo_export_71410037-f1cd-48fb-a6ed-f63e86f313c2.shp'

print(f"Examining shapefile: {shapefile_path}")
print("=" * 70)

# Use ogrinfo to get detailed information
result = subprocess.run(
    ['ogrinfo', str(shapefile_path)],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("Error running ogrinfo. Make sure GDAL is installed:")
    print(result.stderr)
    print("\nInstallation instructions:")
    print("  macOS: brew install gdal")
    print("  Ubuntu: sudo apt-get install gdal-bin")

In [ ]:
# Load the shapefile using GeoPandas
# GeoPandas wraps GDAL/OGR for easier Python data manipulation

logger.info("Loading shapefile into GeoPandas...")
gdf = gpd.read_file(shapefile_path)

print(f"\nDataset Summary:")
print(f"  • Feature count: {len(gdf)}")
print(f"  • Geometry type: {gdf.geometry.type.unique()}")
print(f"  • Coordinate Reference System (CRS): {gdf.crs}")
print(f"  • Spatial extent (bounds):")
bounds = gdf.total_bounds
print(f"    - West: {bounds[0]:.6f}, South: {bounds[1]:.6f}")
print(f"    - East: {bounds[2]:.6f}, North: {bounds[3]:.6f}")

print(f"\nColumn names and types:")
print(gdf.dtypes)

print(f"\nFirst few features:")
print(gdf.head())

## Section 3: Data Validation and Coordinate System Transformation

### Why Transform Coordinates?

Web maps require data in Web Mercator (EPSG:3857) or similar web-friendly projections.
Our data likely uses a local projected coordinate system (like UTM or State Plane).
We need to transform it to a web-compatible system.

**Key points:**
- Original CRS: Likely UTM or local projection
- Target CRS: EPSG:3857 (Web Mercator, used by most web mapping libraries)
- This transformation preserves accuracy while enabling web compatibility

In [ ]:
# Validate the data

logger.info("Validating data integrity...")

# Check for null geometries
null_geoms = gdf.geometry.isnull().sum()
print(f"\nData Validation:")
print(f"  • Null geometries: {null_geoms}")

# Check for invalid geometries
invalid_geoms = (~gdf.geometry.is_valid).sum()
print(f"  • Invalid geometries: {invalid_geoms}")

# Check for empty geometries
empty_geoms = (gdf.geometry.is_empty).sum()
print(f"  • Empty geometries: {empty_geoms}")

# Display current CRS
print(f"\nCurrent Coordinate Reference System:")
print(f"  • EPSG Code: {gdf.crs.to_epsg()}")
print(f"  • CRS Name: {gdf.crs.to_string()}")

# Check if already in Web Mercator
if gdf.crs.to_epsg() == 3857:
    print("\n✓ Data is already in Web Mercator (EPSG:3857)")
    gdf_web = gdf.copy()
else:
    print(f"\n→ Transforming from EPSG:{gdf.crs.to_epsg()} to Web Mercator (EPSG:3857)...")
    gdf_web = gdf.to_crs(epsg=3857)
    print("✓ Transformation complete")
    print(f"  New extent: {gdf_web.total_bounds}")

## Section 4: Convert Vector Data to PMTiles

### The Conversion Pipeline

```
Shapefile → FlatGeoBuf → PMTiles
  ↓          ↓            ↓
OGR Format  Streaming    Cloud-ready
           Vector       Tiles with
           Format       Index
```

### Process Steps

1. **Export to FlatGeoBuf**: Convert shapefile to an intermediate streaming format
   - FlatGeoBuf is efficient and supports large datasets
   - Acts as a bridge format that PMTiles tools can read

2. **Create PMTiles**: Generate the final cloud-optimized format
   - PMTiles bundles all tiles in a single file
   - Includes an internal directory at the start of the file
   - Enables HTTP range requests to fetch only needed tiles

In [ ]:
# Step 1: Export shapefile to FlatGeoBuf format
# FlatGeoBuf is an efficient binary format optimized for vector tiles

logger.info("Converting shapefile to FlatGeoBuf...")

# Save our web-mercator data to shapefile first (will use that as source)
temp_shp = SCRATCH_DIR / 'temp_roads_3857.shp'
gdf_web.to_file(temp_shp)

# Path for FlatGeoBuf intermediate format
flatgeobuf_path = SCRATCH_DIR / 'roads.fgb'

# Use GDAL's ogr2ogr to convert
cmd_fgb = [
    'ogr2ogr',
    '-f', 'FlatGeoBuf',  # Output format
    str(flatgeobuf_path),  # Output file
    str(temp_shp),       # Input file
    '-nln', 'roads'      # Layer name
]

print(f"Running: {' '.join(cmd_fgb)}")
result = subprocess.run(cmd_fgb, capture_output=True, text=True)

if result.returncode == 0:
    print(f"✓ FlatGeoBuf created: {flatgeobuf_path}")
    print(f"  File size: {flatgeobuf_path.stat().st_size / 1024:.2f} KB")
else:
    print("Error:", result.stderr)
    print("\nNote: If ogr2ogr is not found, install GDAL:")
    print("  macOS: brew install gdal")
    print("  Ubuntu: sudo apt-get install gdal-bin")

In [ ]:
# Step 2: Create PMTiles from FlatGeoBuf
# We'll use the command-line approach as it's most robust.

logger.info("Creating PMTiles from FlatGeoBuf...")

pmtiles_path = DATA_DIR / 'santa_clara_roads.pmtiles'

cmd_pmtiles = [
    'tippecanoe',
    '-o', str(pmtiles_path),      # Output PMTiles file
    '-z', '14',                   # Max zoom level (detail level)
    '-Z', '10',                   # Min zoom level (overview level)
    '-l', 'roads',                # Layer name
    '--drop-densest-as-needed',   # Optimize for size
    str(flatgeobuf_path)          # Input FlatGeoBuf file
]

print(f"Command: {' '.join(cmd_pmtiles)}")
result = subprocess.run(cmd_pmtiles, capture_output=True, text=True)

if result.returncode == 0:
    print(f"\n✓ PMTiles created successfully: {pmtiles_path}")
    print(f"  File size: {pmtiles_path.stat().st_size / (1024*1024):.2f} MB")
    print(f"  Layer: roads")
    print(f"  Zoom levels: 10-14")
else:
    print("Error output:", result.stderr)


In [ ]:
# Verify PMTiles file exists and check its properties

logger.info("Verifying PMTiles output...")

if pmtiles_path.exists():
    file_size = pmtiles_path.stat().st_size
    size_mb = file_size / (1024 * 1024)
    size_gb = file_size / (1024 * 1024 * 1024)

    print(f"✓ PMTiles file created successfully!")
    print(f"\n  Path: {pmtiles_path}")
    print(f"  File size: {size_mb:.2f} MB" if size_mb < 1024 else f"  File size: {size_gb:.2f} GB")
    print(f"\n  Metadata:")
    print(f"  • Layer name: roads")
    print(f"  • Source: {shapefile_path.name}")
    print(f"  • Features: {len(gdf_web):,}")
    print(f"  • Geometry type: LineString (roads)")
    print(f"  • Coordinate system: Web Mercator (EPSG:3857)")
    print(f"  • Bounds: {gdf_web.total_bounds}")

    # Get more details using ogrinfo if available
    print(f"\nDetailed layer information:")
    cmd = ['ogrinfo', '-json', str(pmtiles_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout:
        try:
            info = json.loads(result.stdout)
            if 'layers' in info and len(info['layers']) > 0:
                layer = info['layers'][0]
                print(f"  • Feature count: {layer.get('featureCount', 'unknown')}")
        except:
            pass
else:
    print("⚠️  PMTiles file not found at expected path:")
    print(f"   {pmtiles_path}")
    print("\nMake sure tippecanoe is installed and the previous cell ran successfully.")

## Section 5: Convert Raster Data to Cloud-Optimized GeoTIFF

### Why Make a COG?

A Cloud-Optimized GeoTIFF stores raster data in a layout that web tools can read efficiently over HTTP.
That means a map viewer can request only the pieces it needs instead of downloading the whole image.

In this example, we convert `data/santa_clara_1896.tif` into a COG for faster web delivery and zoom-based access.


In [ ]:
# Convert the historical raster into a Cloud-Optimized GeoTIFF.
# We use GDAL's COG output driver so the file is organized for web access.

source_tif = DATA_DIR / 'santa_clara_1896.tif'
cog_path = DATA_DIR / 'santa_clara_1896_cog.tif'

print(f"Source raster: {source_tif}")
print(f"COG output: {cog_path}")

# Remove an older output file first so reruns stay predictable for students.
if cog_path.exists():
    cog_path.unlink()

cmd_cog = [
    'gdal_translate',
    '-of', 'COG',                # Tell GDAL to create a Cloud-Optimized GeoTIFF
    '-co', 'COMPRESS=LZW',       # Compress pixels to reduce file size
    '-co', 'RESAMPLING=AVERAGE', # Build overviews using averaging for smoother zoomed-out views
    '-co', 'LEVEL=9',            # Allow internal pyramid levels for efficient zooming
    str(source_tif),             # Input raster
    str(cog_path)                # Output COG file
]

print(f"Running: {' '.join(cmd_cog)}")
result = subprocess.run(cmd_cog, capture_output=True, text=True)

if result.returncode == 0:
    print(f"\n✓ COG created successfully: {cog_path}")
    print(f"  File size: {cog_path.stat().st_size / (1024*1024):.2f} MB")
else:
    print("Error output:", result.stderr)


In [ ]:
# Verify the Cloud-Optimized GeoTIFF exists and inspect its basic properties.
# This helps students confirm that the conversion worked before they move on.

if cog_path.exists():
    file_size = cog_path.stat().st_size
    size_mb = file_size / (1024 * 1024)
    print("✓ COG file created successfully!")
    print(f"\n  Path: {cog_path}")
    print(f"  File size: {size_mb:.2f} MB")

    with rasterio.open(cog_path) as src:
        # These properties help us confirm that the raster is still georeferenced.
        print("\n  Raster metadata:")
        print(f"  • Width: {src.width}")
        print(f"  • Height: {src.height}")
        print(f"  • Band count: {src.count}")
        print(f"  • CRS: {src.crs}")
        print(f"  • Bounds: {src.bounds}")
        print(f"  • Driver: {src.driver}")
else:
    print("⚠️  COG file not found at expected path:")
    print(f"   {cog_path}")
    print("\nMake sure gdal_translate ran successfully in the previous cell.")


## Useful Resources and References

### Cloud-Optimized Data Formats
- [PMTiles Specification](https://github.com/protomaps/PMTiles)
- [Cloud Optimized GeoTIFF](https://www.cogeo.org/)
- [FlatGeoBuf Format](https://flatgeobuf.org/)

### Tools
- [GDAL/OGR Documentation](https://gdal.org/)
- [Tippecanoe (PMTiles creator)](https://github.com/mapbox/tippecanoe)
- [MapLibre GL JS](https://maplibre.org/)

### Performance Optimization
- [HTTP Range Requests MDN](https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers/Range)
- [Cloud Native Geospatial](https://cloudnativegeo.org/)
- [Stacks Documentation](https://purl.stanford.edu/)

### Data Sources Used in This Workshop
- **Vector**: [Santa Clara County RoadCenterLine](https://data.sccgov.org/Transportation/RoadCenterLine/amyq-ahrs) (Socrata Open Data)
- **Reference**: [Stanford Digital Repository - World PMTiles](https://purl.stanford.edu/hf224mw4004) (108 GB scale example)
- **Historical Maps**: [David Rumsey Map Collection](https://www.davidrumsey.com/luna/servlet/detail/RUMSEY~8~1~1578~170036:San-Mateo,-Santa-Cruz,-Santa-Clara) (Optional raster overlay)

### Additional Data Sources
- [Stanford Digital Repository](https://purl.stanford.edu/)
- [OpenStreetMap Data](https://www.openstreetmap.org/)
- [USGS Data Repository](https://www.usgs.gov/products/maps/gis-data)

---

**Completed!** Your data is ready for web deployment.
Next: Update `index.html` with your GitHub URLs and enable GitHub Pages to go live.